# Parte 2 - Algoritmos no grafo do Slashdot

Este notebook reúne a parte de Teoria dos Grafos da Parte 2: amostragem, BFS, DFS, Dijkstra, Bellman-Ford, escalabilidade e relatório de desempenho. A exploração dos dados (AVD) está em `01_exploracao_dados.ipynb`.

## Pergunta de pesquisa

> No Slashdot, quando um usuário marca outro como amigo ou inimigo, como o tipo de marcação se relaciona com o número mínimo de passos para voltar da pessoa marcada até quem fez a marcação?

## 1. Sobre o dataset

O Slashdot é um site de notícias sobre tecnologia. O recurso Slashdot Zoo permite que um usuário marque outro como amigo ou inimigo. O dataset reúne essas relações.

Vamos utilizar o arquivo `soc-sign-Slashdot090221.txt.gz`, de 21 de fevereiro de 2009, disponível no [SNAP / Stanford](https://snap.stanford.edu/data/soc-sign-Slashdot090221.html).

Cada linha representa uma relação de um usuário para outro. Por isso, a rede é dirigida: uma marcação de A para B não significa que B também marcou A.

| Coluna | Significado |
| --- | --- |
| `origem` | Identificador do usuário que fez a marcação |
| `destino` | Identificador do usuário que recebeu a marcação |
| `sinal` | `1` para amigo e `-1` para inimigo |

## 2. Preparar a leitura

Selecione o Python do ambiente `.venv` do projeto. O bloco abaixo permite importar `ler_dataset`, a função que já criamos no `io.py`. Ele funciona com o notebook aberto pela raiz do projeto ou pela pasta `notebooks`.

In [66]:
# [IA] ChatGPT - apoio na configuracao dos caminhos para importar o projeto
from pathlib import Path
import sys

import pandas as pd

pasta_projeto = Path.cwd()
if pasta_projeto.name == "notebooks":
    pasta_projeto = pasta_projeto.parent

if str(pasta_projeto) not in sys.path:
    sys.path.insert(0, str(pasta_projeto))

from src.graphs.io import ler_dataset

## 3. Ler o arquivo

O arquivo deve estar em `data/dataset_parte2/`. A função `ler_dataset` abre o `.gz`, ignora os comentários e valida os dados antes de devolver a tabela. Não é necessário descompactar.

`dados.head()` mostra as cinco primeiras linhas, para vermos como as informações estão organizadas.

In [67]:
# [IA] ChatGPT - apoio na leitura e exibicao inicial do dataset
caminho_dataset = pasta_projeto / "data" / "dataset_parte2" / "soc-sign-Slashdot090221.txt.gz"
dados = ler_dataset(caminho_dataset)
dados.head()

,origem,destino,sinal
0,0,1,1
1,0,2,1
2,0,3,1
3,0,4,1
4,0,5,1


## 4. Amostragem para os experimentos

A exploração utiliza a rede completa; os experimentos usam recortes com 10%, 50% e 100% dos usuários. Os identificadores são ordenados e embaralhados com a semente 42, e cada recorte usa o início da mesma lista, mantendo os menores contidos nos maiores. O subgrafo induzido preserva os nós selecionados e os arcos entre eles, com sua direção e sinal.

Os percentuais se referem aos nós, não aos arcos; 100% corresponde à rede original. Os recortes podem ficar desconectados, e seus graus, caminhos e ciclos não devem ser generalizados para a rede completa. A estratégia deve ser conferida com a proposta aprovada e com a professora.

In [78]:
# [IA] ChatGPT - apoio na implementacao dos recortes reproduziveis
import random

from src.graphs.io import montar_grafo_dataset

semente = 42
percentuais = [10, 50, 100]
grafo_completo = montar_grafo_dataset(dados)

usuarios_sorteados = sorted(grafo_completo.nos())
random.Random(semente).shuffle(usuarios_sorteados)

recortes = {}
resumo_recortes = []

for percentual in percentuais:
    quantidade = grafo_completo.ordem() * percentual // 100
    recorte = grafo_completo.subgrafo_induzido(usuarios_sorteados[:quantidade])

    recortes[percentual] = recorte
    resumo_recortes.append({
        "usuarios (%)": percentual,
        "nos": recorte.ordem(),
        "arcos": recorte.tamanho(),
        "arcos (%)": round(recorte.tamanho() / grafo_completo.tamanho() * 100, 2),
    })

pd.DataFrame(resumo_recortes).set_index("usuarios (%)")

,nos,arcos,arcos (%)
usuarios (%),,,
10,8214,5434,0.99
50,41070,134558,24.50
100,82140,549202,100.00


**Interpretação**

O recorte de 10% preservou 5.434 arcos (0,99% do total). O de 50% preservou 134.558 (24,50%). Um arco só entra no recorte quando seus dois usuários foram selecionados. Por isso, os percentuais de usuários e de arcos são diferentes. O recorte de 100% preservou os 549.202 arcos.

## 5. Buscas BFS e DFS no Slashdot

Nesta seção, usamos as duas buscas para duas tarefas. Primeiro, verificamos se existe caminho de volta para algumas marcações na rede completa. Depois, medimos quantos usuários as buscas alcançam e quanto tempo levam nos recortes de 10%, 50% e 100%.

Se A marcou B, a volta começa em B e procura A seguindo o sentido das setas. O tipo, amigo ou inimigo, é o da marcação original. Tanto as setas de amigo quanto as de inimigo podem fazer parte da volta, e cada seta conta como um passo.

### 5.1 BFS — Busca em largura

A BFS procura em camadas. Primeiro visita quem está a um passo, depois quem está a dois, e assim por diante. A primeira vez que encontra quem fez a marcação indica a menor distância de volta.

Sorteamos cinco marcações de amigo e cinco de inimigo, sem reposição e com semente 42. Esses dez casos ajudam a conferir a medição.

**Como ler a tabela**

- **Marcação original** mostra quem marcou → quem recebeu.
- **Tipo** indica se a marcação original foi de amigo ou inimigo.
- **Menor caminho de volta** mostra a sequência de usuários, de quem recebeu até quem marcou.
- **Passos** conta as setas do caminho de volta. Se não houver caminho, aparece um traço.

In [86]:
# [IA] ChatGPT - apoio no uso da BFS do projeto e na tabela do piloto
from src.graphs.algorithms import bfs, montar_caminho

marcacoes_piloto = pd.concat([
    dados.loc[dados["sinal"] == 1].sample(n=5, random_state=42),
    dados.loc[dados["sinal"] == -1].sample(n=5, random_state=42),
], ignore_index=True)

resultados_retorno = []
for origem, destino, sinal in marcacoes_piloto.itertuples(index=False, name=None):
    _, pais, _ = bfs(grafo_completo, destino)
    caminho = montar_caminho(pais, origem)
    resultados_retorno.append({
        "Marcação original": f"{origem} → {destino}",
        "Tipo": "Amigo" if sinal == 1 else "Inimigo",
        "Menor caminho de volta": " → ".join(map(str, caminho)) if caminho else "Sem caminho",
        "Passos": len(caminho) - 1 if caminho else None,
    })

piloto_retorno = pd.DataFrame(resultados_retorno)
piloto_retorno["Passos"] = piloto_retorno["Passos"].astype("Int64")
piloto_retorno.assign(Passos=piloto_retorno["Passos"].astype("string").fillna("—"))

,Marcação original,Tipo,Menor caminho de volta,Passos
0,8068 → 48080,Amigo,Sem caminho,—
1,653 → 19563,Amigo,19563 → 6712 → 21537 → 653,3
2,3451 → 9765,Amigo,9765 → 1417 → 3451,2
3,21299 → 134,Amigo,134 → 8 → 50 → 1723 → 21299,4
4,7733 → 11021,Amigo,11021 → 1935 → 686 → 7733,3
5,19236 → 16592,Inimigo,16592 → 68 → 1870 → 19236,3
6,3381 → 29495,Inimigo,29495 → 18596 → 11810 → 3217 → 3245 → 3381,5
7,19094 → 3078,Inimigo,3078 → 19094,1
8,65476 → 277,Inimigo,277 → 191 → 330 → 8193 → 53864 → 65476,5
9,47238 → 36599,Inimigo,36599 → 7191 → 662 → 3338 → 47238,4


**Interpretação**

Para a marcação de amigo **3451 → 9765**, o menor caminho de volta é **9765 → 1417 → 3451**. São dois passos porque há duas setas e uma pessoa intermediária. A primeira seta da volta é de inimigo e a segunda, de amigo.

Das dez marcações sorteadas, nove tiveram volta e uma ficou sem caminho. Essa contagem descreve apenas o piloto. A comparação entre amigo e inimigo precisa de mais casos.

### 5.2 DFS — Busca em profundidade

A DFS segue um ramo até onde consegue. Quando ele acaba, volta ao último ponto com uma opção por explorar. Assim como a BFS, ela encontra o destino se ele estiver ao alcance da fonte. O caminho que ela monta depende da ordem dos vizinhos.

Usamos duas marcações da tabela anterior, uma com volta e outra sem. A coluna “Há retorno?” indica se ambas encontraram um caminho. As outras colunas mostram quantos passos cada caminho tem. Para responder à pergunta de pesquisa, usamos a distância da BFS, que garante a menor volta.

In [87]:
# [IA] ChatGPT - apoio no uso da DFS do projeto na comparação com a BFS
from src.graphs.algorithms import dfs

comparacao_buscas = []
# Linhas do piloto com retorno (2) e sem retorno (0).
for exemplo in (2, 0):
    origem, destino, _ = marcacoes_piloto.loc[exemplo]
    pais_dfs, _, _, _ = dfs(grafo_completo, origem=destino)
    caminho_dfs = montar_caminho(pais_dfs, origem)
    passos_bfs = piloto_retorno.loc[exemplo, "Passos"]
    assert (caminho_dfs is not None) == pd.notna(passos_bfs)

    comparacao_buscas.append({
        "Marcação original": f"{origem} → {destino}",
        "Há retorno?": "Sim" if caminho_dfs else "Não",
        "BFS (passos)": passos_bfs,
        "DFS (passos)": len(caminho_dfs) - 1 if caminho_dfs else None,
    })

tabela_buscas = pd.DataFrame(comparacao_buscas)
colunas_passos = ["BFS (passos)", "DFS (passos)"]
tabela_buscas[colunas_passos] = tabela_buscas[colunas_passos].astype("Int64")
tabela_buscas.astype({coluna: "string" for coluna in colunas_passos}).fillna("—")

,Marcação original,Há retorno?,BFS (passos),DFS (passos)
0,3451 → 9765,Sim,2,775
1,8068 → 48080,Não,—,—


**Interpretação**

As duas buscas concordaram nos dois exemplos sobre a existência de retorno. Na marcação 3451 → 9765, a BFS encontrou a volta em 2 passos e a DFS registrou um caminho de 775. Os dois caminhos chegam ao mesmo usuário. Para medir a menor distância, usamos os 2 passos garantidos pela BFS. O caminho da DFS depende da ordem em que ela explora os vizinhos.

### 5.3 Comparação nas amostragens de 10%, 50% e 100%

Nos blocos 5.1 e 5.2, usamos a rede completa para medir o caminho de volta de marcações específicas. Aqui, a pergunta é como o tamanho do grafo afeta o alcance e o tempo das buscas. Usamos os recortes de 10%, 50% e 100% dos usuários criados na seção 4.

Para comparar os três tamanhos, usamos os mesmos três usuários de partida. Sorteamos os pontos de partida entre os usuários do recorte de 10% que têm pelo menos uma seta de saída nesse recorte, com semente 42. Como os recortes menores estão contidos nos maiores, esses usuários aparecem nos três tamanhos.

In [88]:
# [IA] ChatGPT - apoio no sorteio reproduzível das fontes
candidatos = sorted(
    usuario for usuario in recortes[10].nos()
    if recortes[10].grau_saida(usuario) > 0
)
fontes_buscas = random.Random(42).sample(candidatos, 3)

#### 5.3.1 Medição das buscas

Executamos cada busca cinco vezes para cada usuário de partida e cada recorte. A tabela mostra o tempo mediano: o valor do meio quando ordenamos as cinco medições. Medimos apenas as chamadas de BFS e DFS. A montagem dos recortes e a conferência dos resultados ficam fora do cronômetro.

**Como ler as colunas**

- **Usuário de partida** é o ID da pessoa de onde a busca começa.
- **Máximo de passos (BFS)** mostra o maior número de passos entre os menores caminhos do usuário de partida até os usuários alcançados. Cada seta conta como um passo.
- **Máximo de passos (DFS)** conta os passos do ramo mais profundo registrado pela DFS a partir do usuário de partida. A DFS segue um ramo de cada vez, por isso esse caminho pode ter bem mais passos que o menor caminho até o mesmo usuário.

In [89]:
# [IA] ChatGPT - apoio na medição da BFS e da DFS nos recortes
from collections import Counter
from statistics import median
from time import perf_counter

from src.graphs.algorithms import bfs, dfs

buscas = [("BFS", bfs), ("DFS", dfs)]
linhas_escalabilidade = []
for percentual, grafo in recortes.items():
    for fonte in fontes_buscas:
        tempos = {"BFS": [], "DFS": []}
        for repeticao in range(5):
            ordem_buscas = buscas if repeticao % 2 == 0 else buscas[::-1]
            for nome, busca in ordem_buscas:
                inicio = perf_counter()
                resultado = busca(grafo, fonte)
                tempos[nome].append((perf_counter() - inicio) * 1000)
                if nome == "BFS":
                    distancias, _, ordem_bfs = resultado
                else:
                    pais_dfs, ordem_dfs, _, tem_ciclo = resultado
                del resultado
            assert set(ordem_bfs) == set(ordem_dfs), "As buscas alcançaram usuários diferentes"

        # Na ordem de visita, o usuário anterior no caminho já tem profundidade calculada.
        passos_dfs = {fonte: 0}
        for usuario in ordem_dfs[1:]:
            passos_dfs[usuario] = passos_dfs[pais_dfs[usuario]] + 1

        linhas_escalabilidade.append({
            "Recorte (%)": percentual,
            "Nós": grafo.ordem(),
            "Arcos": grafo.tamanho(),
            "Usuário de partida": fonte,
            "Alcançados": len(ordem_bfs),
            "Máximo de passos (BFS)": max(distancias.values()),
            "Máximo de passos (DFS)": max(passos_dfs.values()),
            "Ciclo (DFS)": "Sim" if tem_ciclo else "Não",
            "BFS (ms)": median(tempos["BFS"]),
            "DFS (ms)": median(tempos["DFS"]),
            "Usuários por camada BFS": dict(sorted(Counter(distancias.values()).items())),
            "Primeiros 10 da BFS": ordem_bfs[:10],
            "Primeiros 10 da DFS": ordem_dfs[:10],
            "Tempos BFS por execução": tempos["BFS"],
            "Tempos DFS por execução": tempos["DFS"],
        })

escalabilidade_buscas = pd.DataFrame(linhas_escalabilidade)
resumo_buscas = escalabilidade_buscas[[
    "Recorte (%)", "Usuário de partida",
    "Máximo de passos (BFS)", "Máximo de passos (DFS)", "BFS (ms)", "DFS (ms)",
]].round({"BFS (ms)": 3, "DFS (ms)": 3})
resumo_buscas

,Recorte (%),Usuário de partida,Máximo de passos (BFS),Máximo de passos (DFS),BFS (ms),DFS (ms)
0,10,48088,10,108,3.356,5.654
1,10,4710,1,1,0.005,0.006
2,10,1199,9,111,2.850,4.502
3,50,48088,9,2232,50.890,149.745
4,50,4710,8,2205,47.724,142.367
5,50,1199,8,2223,37.264,157.010
6,100,48088,9,6636,280.445,1184.657
7,100,4710,9,6591,213.303,567.667
8,100,1199,8,6498,136.621,516.786


**Interpretação**

As duas buscas alcançaram os mesmos usuários nos nove casos. Mesmo assim, os caminhos registrados têm tamanhos diferentes. Na rede completa, partindo de 48088, todos os usuários alcançáveis estavam a até 9 passos pela BFS. A DFS registrou um ramo de 6.636 passos. Os dois máximos podem corresponder a usuários diferentes.

As medianas de tempo aumentaram com o tamanho dos recortes para os três usuários testados. A medição da DFS inclui a classificação dos arcos e a detecção de ciclos. A tabela detalhada `escalabilidade_buscas` guarda também o total de alcançados, as camadas da BFS, a ordem inicial de visita, os ciclos e os tempos individuais.

## 6. Dijkstra no Slashdot

O Dijkstra não aceita pesos negativos. Por isso, ele não pode usar diretamente os sinais +1 e −1 do dataset (seção 4.6 do notebook `01_exploracao_dados.ipynb`). Nesta seção, transformamos os pesos assim:

| Marcação | Sinal no dataset | Peso no Dijkstra |
| --- | --- | --- |
| Amigo | +1 | 1 |
| Inimigo | −1 | 2 |

Cada seta continua valendo pelo menos um passo, e a seta de inimigo custa o dobro. Assim, o custo de um caminho é o número de passos somado ao número de setas de inimigo. Um passo a mais custa o mesmo que uma seta de inimigo, então o Dijkstra pode escolher uma volta mais longa se ela usar menos setas de inimigo.

Essa transformação muda os caminhos mínimos: o caminho de menor custo pode ter mais passos que o caminho da BFS. A célula abaixo cria, para cada recorte da seção 4, uma cópia do grafo com esses pesos. Os arcos, a direção e a ordem dos vizinhos continuam os mesmos.

In [90]:
# [IA] Claude – transformação dos pesos para o Dijkstra (amigo 1, inimigo 2)
from collections import Counter

from src.graphs.graph import GrafoDirigido

PESO_DIJKSTRA = {1: 1, -1: 2}


def grafo_com_pesos_dijkstra(grafo):
    novo = GrafoDirigido(grafo.nome + "_dijkstra")
    for usuario in grafo.nos():
        novo.adicionar_no(usuario)
    for origem in grafo.nos():
        for destino, sinal, _, _ in grafo.vizinhos(origem):
            novo.adicionar_arco(origem, destino, peso=PESO_DIJKSTRA[sinal])
    return novo


recortes_dijkstra = {percentual: grafo_com_pesos_dijkstra(grafo) for percentual, grafo in recortes.items()}

# conferência: na rede completa, os pesos 1 e 2 devem repetir a contagem de amigos e inimigos da seção 4.6 do notebook 01_exploracao_dados
contagem_pesos = Counter(
    peso
    for usuario in recortes_dijkstra[100].nos()
    for _, peso, _, _ in recortes_dijkstra[100].vizinhos(usuario)
)
pd.Series(
    {
        "arcos com peso 1 (amigo)": contagem_pesos[1],
        "arcos com peso 2 (inimigo)": contagem_pesos[2],
        "arcos com peso negativo": sum(n for peso, n in contagem_pesos.items() if peso < 0),
    },
    name="quantidade",
).to_frame()

,quantidade
arcos com peso 1 (amigo),425072
arcos com peso 2 (inimigo),124130
arcos com peso negativo,0


### 6.1 Volta das marcações do piloto

Usamos as mesmas dez marcações da seção 5.1, cinco de amigo e cinco de inimigo. Para cada marcação A → B, o Dijkstra começa em B e calcula o menor custo de volta até A na rede completa. Assim, comparamos a volta de menor custo com a volta de menos passos encontrada pela BFS.

Executamos o Dijkstra cinco vezes para cada marcação e mostramos o tempo médio. Medimos apenas as chamadas do Dijkstra. Cada chamada calcula o custo de B até todos os usuários que ele alcança, e não só até A.

**Como ler as colunas**

- **Passos (BFS)** repete o menor número de passos da seção 5.1.
- **Custo (Dijkstra)** é a soma dos pesos da volta de menor custo.
- **Passos (Dijkstra)** conta as setas dessa volta.
- **Setas de inimigo** conta quantas setas de inimigo a volta usa. Como cada seta de inimigo vale 2, esse número é o custo menos os passos.
- **Volta pelo Dijkstra** mostra a sequência de usuários, de quem recebeu a marcação até quem a fez.
- **Tempo médio (ms)** é a média das cinco execuções, em milissegundos.

In [91]:
# [IA] Claude – Dijkstra nas marcações do piloto, com conferência e tempo médio
from statistics import mean, median
from time import perf_counter

from src.graphs.algorithms import dijkstra, montar_caminho

grafo_dijkstra = recortes_dijkstra[100]
linhas_dijkstra = []
for (origem, destino, sinal), passos_bfs in zip(
    marcacoes_piloto.itertuples(index=False, name=None), piloto_retorno["Passos"]
):
    tempos = []
    for _ in range(5):
        inicio = perf_counter()
        distancias, pais, _ = dijkstra(grafo_dijkstra, destino)
        tempos.append((perf_counter() - inicio) * 1000)

    caminho = montar_caminho(pais, origem)
    # o Dijkstra e a BFS precisam concordar sobre a existência da volta
    assert (caminho is not None) == pd.notna(passos_bfs)
    custo = distancias[origem] if caminho else None
    passos = len(caminho) - 1 if caminho else None

    linhas_dijkstra.append({
        "Marcação original": f"{origem} → {destino}",
        "Tipo": "Amigo" if sinal == 1 else "Inimigo",
        "Passos (BFS)": passos_bfs,
        "Custo (Dijkstra)": custo,
        "Passos (Dijkstra)": passos,
        "Setas de inimigo": custo - passos if caminho else None,
        "Volta pelo Dijkstra": " → ".join(map(str, caminho)) if caminho else "Sem caminho",
        "Tempo médio (ms)": mean(tempos),
        "Tempos por execução (ms)": tempos,
    })

pares_dijkstra = pd.DataFrame(linhas_dijkstra)
colunas_inteiras = ["Passos (BFS)", "Custo (Dijkstra)", "Passos (Dijkstra)", "Setas de inimigo"]
pares_dijkstra[colunas_inteiras] = pares_dijkstra[colunas_inteiras].astype("Int64")
(
    pares_dijkstra.drop(columns="Tempos por execução (ms)")
    .astype({coluna: "string" for coluna in colunas_inteiras})
    .fillna("—")
    .round({"Tempo médio (ms)": 1})
)

,Marcação original,Tipo,Passos (BFS),Custo (Dijkstra),Passos (Dijkstra),Setas de inimigo,Volta pelo Dijkstra,Tempo médio (ms)
0,8068 → 48080,Amigo,—,—,—,—,Sem caminho,1.4
1,653 → 19563,Amigo,3,3,3,0,19563 → 6712 → 21537 → 653,344.3
2,3451 → 9765,Amigo,2,2,2,0,9765 → 4204 → 3451,318.3
3,21299 → 134,Amigo,4,4,4,0,134 → 8 → 83 → 2920 → 21299,319.0
4,7733 → 11021,Amigo,3,4,3,1,11021 → 1935 → 686 → 7733,288.8
5,19236 → 16592,Inimigo,3,3,3,0,16592 → 197 → 5199 → 19236,281.0
6,3381 → 29495,Inimigo,5,6,6,0,29495 → 18596 → 11810 → 79 → 170 → 23840 → 3381,290.6
7,19094 → 3078,Inimigo,1,2,1,1,3078 → 19094,281.6
8,65476 → 277,Inimigo,5,6,5,1,277 → 191 → 330 → 8193 → 53864 → 65476,275.3
9,47238 → 36599,Inimigo,4,4,4,0,36599 → 7191 → 662 → 3338 → 47238,282.2


**Interpretação**

_A escrever pelo grupo._

## 7. Bellman-Ford no Slashdot

O Bellman-Ford aceita pesos negativos, então aqui usamos os sinais originais do dataset: +1 para amigo e −1 para inimigo. Com esses pesos, o custo de um caminho é o número de setas de amigo menos o número de setas de inimigo.

Um ciclo negativo é um ciclo com mais setas de inimigo do que de amigo. O menor exemplo é A → B → A quando os dois usuários se marcaram como inimigos: o custo do ciclo é −2. Se um usuário consegue chegar a um ciclo negativo, o menor custo até os usuários alcançáveis a partir desse ciclo deixa de existir, porque cada volta extra no ciclo diminui o custo.

A implementação do projeto confere, a cada rodada, se os ponteiros de pai formam um ciclo. Quando isso acontece, ela para e devolve o ciclo encontrado.

Para mostrar os dois casos, usamos o mesmo usuário de partida em dois tamanhos do grafo:

1. No recorte de 10%, a partir dele há setas de inimigo, mas nenhum ciclo negativo. O Bellman-Ford termina e calcula todos os custos.
2. Na rede completa, a partir do mesmo usuário, o Bellman-Ford encontra um ciclo negativo.

Para escolher esse usuário, rodamos o Bellman-Ford a partir de cada usuário do recorte de 10%. Ficamos com os que não chegaram a um ciclo negativo e alcançaram pelo menos um custo negativo. Entre eles, escolhemos o que alcança mais usuários (no empate, o menor ID) e que, na rede completa, chega a um ciclo negativo.

In [92]:
# [IA] Claude – escolha do usuário de partida para os dois casos do Bellman-Ford
from src.graphs.algorithms import bellman_ford, bfs, custo_caminho

candidatos_bf = []
for usuario in sorted(recortes[10].nos()):
    distancias, _, ciclo = bellman_ford(recortes[10], usuario)
    if ciclo is None and min(distancias.values()) < 0:
        candidatos_bf.append((usuario, len(distancias)))

# mais usuários alcançados primeiro; no empate, o menor ID
candidatos_bf.sort(key=lambda candidato: (-candidato[1], candidato[0]))
fonte_bf = next(
    usuario for usuario, _ in candidatos_bf
    if bellman_ford(recortes[100], usuario)[2] is not None
)
print(f"Usuários do recorte de 10% sem ciclo negativo e com algum custo negativo: {len(candidatos_bf)}")
print(f"Usuário de partida escolhido: {fonte_bf}")

Usuários do recorte de 10% sem ciclo negativo e com algum custo negativo: 136
Usuário de partida escolhido: 11038


### 7.1 Caso com peso negativo e sem ciclo negativo (recorte de 10%)

Rodamos o Bellman-Ford cinco vezes a partir do usuário escolhido, no recorte de 10% e com os pesos originais. A tabela mostra cada usuário alcançado, o custo calculado e o caminho de menor custo, com o sinal de cada seta. Antes da tabela aparecem o total de alcançados, as setas de inimigo que saem desses usuários e o tempo médio.

**Como ler as colunas**

- **Custo** é a soma dos sinais do caminho: cada seta de amigo soma 1 e cada seta de inimigo subtrai 1.
- **Caminho** mostra a sequência de usuários a partir do usuário de partida.
- **Sinais** mostra o sinal de cada seta do caminho, na mesma ordem.

In [93]:
# [IA] Claude – Bellman-Ford sem ciclo negativo no recorte de 10%
def sinal_do_arco(grafo, origem, destino):
    return next(peso for vizinho, peso, _, _ in grafo.vizinhos(origem) if vizinho == destino)


def medir_bellman_ford(grafo, fonte, repeticoes=5):
    tempos = []
    for _ in range(repeticoes):
        inicio = perf_counter()
        resultado = bellman_ford(grafo, fonte)
        tempos.append((perf_counter() - inicio) * 1000)
    return resultado, tempos


(distancias_bf, pais_bf, ciclo_bf), tempos_bf_sem_ciclo = medir_bellman_ford(recortes[10], fonte_bf)
assert ciclo_bf is None

linhas_bf = []
for usuario, custo in sorted(distancias_bf.items(), key=lambda item: (item[1], item[0])):
    caminho = montar_caminho(pais_bf, usuario)
    sinais = [sinal_do_arco(recortes[10], u, v) for u, v in zip(caminho, caminho[1:])]
    assert sum(sinais) == custo  # o custo calculado bate com a soma dos sinais do caminho
    linhas_bf.append({
        "Usuário": usuario,
        "Custo": custo,
        "Caminho": " → ".join(map(str, caminho)),
        "Sinais": " ".join(f"{sinal:+d}" for sinal in sinais) or "—",
    })

setas_inimigo_alcancaveis = sum(
    1 for usuario in distancias_bf for _, peso, _, _ in recortes[10].vizinhos(usuario) if peso < 0
)
print(f"Usuários alcançados: {len(distancias_bf)}")
print(f"Setas de inimigo saindo dos usuários alcançados: {setas_inimigo_alcancaveis}")
print("Ciclo negativo: não encontrado")
print(f"Tempo médio de 5 execuções: {mean(tempos_bf_sem_ciclo):.3f} ms")
pd.DataFrame(linhas_bf)

Usuários alcançados: 13
Setas de inimigo saindo dos usuários alcançados: 6
Ciclo negativo: não encontrado
Tempo médio de 5 execuções: 0.043 ms


,Usuário,Custo,Caminho,Sinais
0,0,-1,11038 → 0,-1
1,9,-1,11038 → 9,-1
2,90,-1,11038 → 90,-1
3,2210,-1,11038 → 2210,-1
4,2224,-1,11038 → 2224,-1
5,46452,-1,11038 → 46452,-1
6,18,0,11038 → 0 → 18,-1 +1
7,56,0,11038 → 9 → 56,-1 +1
8,11038,0,11038,—
9,11072,1,11038 → 9 → 56 → 11072,-1 +1 +1


**Interpretação**

_A escrever pelo grupo._

### 7.2 Caso com ciclo negativo (rede completa)

Agora rodamos o Bellman-Ford cinco vezes a partir do mesmo usuário, na rede completa e com os pesos originais. O algoritmo para quando encontra um ciclo negativo. A tabela mostra as setas do ciclo devolvido, e a soma dos sinais confirma que o custo do ciclo é negativo.

Para mostrar que o usuário de partida chega ao ciclo, também usamos a BFS na rede completa para montar o menor caminho dele até um dos usuários do ciclo.

In [94]:
# [IA] Claude – Bellman-Ford com ciclo negativo na rede completa
(_, _, ciclo_negativo), tempos_bf_com_ciclo = medir_bellman_ford(recortes[100], fonte_bf)
assert ciclo_negativo is not None

custo_ciclo = custo_caminho(recortes[100], ciclo_negativo)
assert custo_ciclo < 0

_, pais_ate_ciclo, _ = bfs(recortes[100], fonte_bf)
caminho_ate_ciclo = min(
    (montar_caminho(pais_ate_ciclo, usuario) for usuario in ciclo_negativo[:-1]),
    key=len,
)

print(f"Ciclo encontrado: {' → '.join(map(str, ciclo_negativo))}")
print(f"Custo do ciclo: {custo_ciclo}")
print(f"Caminho do usuário de partida até o ciclo (BFS): {' → '.join(map(str, caminho_ate_ciclo))}")
print(f"Tempo médio de 5 execuções: {mean(tempos_bf_com_ciclo):.3f} ms")

pd.DataFrame([
    {
        "De": u,
        "Para": v,
        "Marcação": "Amigo" if sinal_do_arco(recortes[100], u, v) == 1 else "Inimigo",
        "Sinal": sinal_do_arco(recortes[100], u, v),
    }
    for u, v in zip(ciclo_negativo, ciclo_negativo[1:])
])

Ciclo encontrado: 919 → 949 → 919
Custo do ciclo: -2
Caminho do usuário de partida até o ciclo (BFS): 11038 → 8 → 919
Tempo médio de 5 execuções: 8.767 ms


,De,Para,Marcação,Sinal
0,919,949,Inimigo,-1
1,949,919,Inimigo,-1


**Interpretação**

_A escrever pelo grupo._

## 8. Escalabilidade dos quatro algoritmos

### 8.1 Dijkstra e Bellman-Ford nos recortes de 10%, 50% e 100%

Repetimos a medição da seção 5.3 para o Dijkstra e o Bellman-Ford, com os mesmos três usuários de partida e os mesmos recortes. Nesta medição, os dois algoritmos usam os pesos do Dijkstra (amigo 1, inimigo 2). Assim, o Bellman-Ford não para em um ciclo negativo e calcula os custos até o fim, e podemos conferir se os dois chegam aos mesmos custos.

Executamos cada algoritmo cinco vezes para cada usuário de partida e cada recorte, alternando a ordem entre os dois a cada repetição. A tabela mostra o tempo médio das cinco execuções. Medimos apenas as chamadas dos algoritmos.

**Como ler as colunas**

- **Alcançados** é o número de usuários com custo calculado a partir do usuário de partida, incluindo ele mesmo.
- **Maior custo** é o maior custo de menor caminho entre os usuários alcançados.
- **Mesmos custos?** indica se o Dijkstra e o Bellman-Ford calcularam exatamente os mesmos custos para todos os usuários alcançados.
- **Dijkstra (ms)** e **Bellman-Ford (ms)** são os tempos médios de cinco execuções, em milissegundos.

In [95]:
# [IA] Claude – medição do Dijkstra e do Bellman-Ford nos recortes
algoritmos_caminhos = [("Dijkstra", dijkstra), ("Bellman-Ford", bellman_ford)]
linhas_caminhos = []
for percentual, grafo in recortes_dijkstra.items():
    for fonte in fontes_buscas:
        tempos = {"Dijkstra": [], "Bellman-Ford": []}
        for repeticao in range(5):
            ordem_algoritmos = algoritmos_caminhos if repeticao % 2 == 0 else algoritmos_caminhos[::-1]
            for nome, algoritmo in ordem_algoritmos:
                inicio = perf_counter()
                resultado = algoritmo(grafo, fonte)
                tempos[nome].append((perf_counter() - inicio) * 1000)
                if nome == "Dijkstra":
                    custos_dijkstra = resultado[0]
                else:
                    custos_bf, _, ciclo = resultado
                    assert ciclo is None  # com pesos 1 e 2 não existe ciclo negativo
                del resultado

        linhas_caminhos.append({
            "Recorte (%)": percentual,
            "Nós": grafo.ordem(),
            "Arcos": grafo.tamanho(),
            "Usuário de partida": fonte,
            "Alcançados": len(custos_dijkstra),
            "Maior custo": max(custos_dijkstra.values()),
            "Mesmos custos?": "Sim" if custos_dijkstra == custos_bf else "Não",
            "Dijkstra (ms)": mean(tempos["Dijkstra"]),
            "Bellman-Ford (ms)": mean(tempos["Bellman-Ford"]),
            "Tempos Dijkstra por execução": tempos["Dijkstra"],
            "Tempos Bellman-Ford por execução": tempos["Bellman-Ford"],
        })

escalabilidade_caminhos = pd.DataFrame(linhas_caminhos)
escalabilidade_caminhos[[
    "Recorte (%)", "Usuário de partida", "Alcançados", "Maior custo", "Mesmos custos?",
    "Dijkstra (ms)", "Bellman-Ford (ms)",
]].round({"Dijkstra (ms)": 3, "Bellman-Ford (ms)": 3})

,Recorte (%),Usuário de partida,Alcançados,Maior custo,Mesmos custos?,Dijkstra (ms),Bellman-Ford (ms)
0,10,48088,1412,13,Sim,3.158,7.193
1,10,4710,2,1,Sim,0.002,0.003
2,10,1199,1411,11,Sim,3.388,5.699
3,50,48088,23123,11,Sim,75.820,146.704
4,50,4710,23120,11,Sim,104.657,142.030
5,50,1199,23120,10,Sim,77.542,122.493
6,100,48088,67272,10,Sim,272.656,396.083
7,100,4710,67266,11,Sim,296.407,401.774
8,100,1199,67266,10,Sim,270.713,399.492


**Interpretação**

_A escrever pelo grupo._

### 8.2 Crescimento do tempo e complexidade teórica

A tabela junta as medições da seção 5.3 (BFS e DFS) e da seção 8.1 (Dijkstra e Bellman-Ford). Para cada algoritmo e recorte, calculamos a média de todas as execuções dos três usuários de partida. A seção 5.3 mostra a mediana; aqui usamos a média das mesmas execuções, que é a medida pedida no enunciado. Depois dividimos o tempo de cada recorte pelo tempo do recorte de 10%, para ver quantas vezes ele cresceu.

O crescimento previsto faz a mesma divisão com a complexidade teórica de cada algoritmo, usando o número de nós (V) e de arcos (E) de cada recorte:

| Algoritmo | Complexidade usada |
| --- | --- |
| BFS e DFS | V + E |
| Dijkstra (com heap) | (V + E) · log V |
| Bellman-Ford | V · E |

A complexidade descreve o pior caso no grafo inteiro. Os algoritmos, porém, só percorrem a parte do grafo que o usuário de partida alcança, e o Bellman-Ford do projeto para assim que uma rodada não muda nenhum custo.

**Como ler as colunas**

- **Tempo médio (ms)** é a média de todas as execuções dos três usuários de partida naquele recorte.
- **Crescimento medido (×)** é o tempo médio do recorte dividido pelo tempo médio do recorte de 10%.
- **Crescimento previsto (×)** é a complexidade do recorte dividida pela complexidade do recorte de 10%.

In [96]:
# [IA] Claude – crescimento medido × crescimento previsto pela complexidade
import math

complexidades = {
    "BFS": lambda v, e: v + e,
    "DFS": lambda v, e: v + e,
    "Dijkstra": lambda v, e: (v + e) * math.log2(v),
    "Bellman-Ford": lambda v, e: v * e,
}
tempos_por_algoritmo = {
    "BFS": (escalabilidade_buscas, "Tempos BFS por execução"),
    "DFS": (escalabilidade_buscas, "Tempos DFS por execução"),
    "Dijkstra": (escalabilidade_caminhos, "Tempos Dijkstra por execução"),
    "Bellman-Ford": (escalabilidade_caminhos, "Tempos Bellman-Ford por execução"),
}

linhas_crescimento = []
for algoritmo, (tabela, coluna) in tempos_por_algoritmo.items():
    base = None
    for percentual, grupo in tabela.groupby("Recorte (%)"):
        v, e = recortes[percentual].ordem(), recortes[percentual].tamanho()
        tempo = mean(t for tempos in grupo[coluna] for t in tempos)
        complexidade = complexidades[algoritmo](v, e)
        if base is None:  # o primeiro grupo é o recorte de 10%
            base = (tempo, complexidade)
        linhas_crescimento.append({
            "Algoritmo": algoritmo,
            "Recorte (%)": percentual,
            "Nós (V)": v,
            "Arcos (E)": e,
            "Tempo médio (ms)": tempo,
            "Crescimento medido (×)": tempo / base[0],
            "Crescimento previsto (×)": complexidade / base[1],
        })

crescimento_tempos = pd.DataFrame(linhas_crescimento)
crescimento_tempos.round({"Tempo médio (ms)": 3, "Crescimento medido (×)": 1, "Crescimento previsto (×)": 1})

,Algoritmo,Recorte (%),Nós (V),Arcos (E),Tempo médio (ms),Crescimento medido (×),Crescimento previsto (×)
0,BFS,10,8214,5434,2.183,1.0,1.0
1,BFS,50,41070,134558,42.875,19.6,12.9
2,BFS,100,82140,549202,232.661,106.6,46.3
3,DFS,10,8214,5434,3.723,1.0,1.0
4,DFS,50,41070,134558,147.612,39.6,12.9
5,DFS,100,82140,549202,815.498,219.0,46.3
6,Dijkstra,10,8214,5434,2.183,1.0,1.0
7,Dijkstra,50,41070,134558,86.007,39.4,15.2
8,Dijkstra,100,82140,549202,279.925,128.2,58.1
9,Bellman-Ford,10,8214,5434,4.299,1.0,1.0


**Interpretação**

_A escrever pelo grupo._

## 9. Relatório de desempenho

A célula abaixo salva em `out/parte2/parte2_report.json` todas as medições de tempo desta parte: BFS e DFS nos recortes (seção 5.3), Dijkstra nas marcações do piloto (6.1), Bellman-Ford sem e com ciclo negativo (7.1 e 7.2) e Dijkstra e Bellman-Ford nos recortes (8.1). Cada medição guarda o algoritmo, a tarefa, o recorte, o usuário de partida, os pesos usados, os tempos das cinco execuções, a média e a mediana.

In [97]:
# [IA] Claude – montagem e gravação do parte2_report.json
import json


def medicao(algoritmo, tarefa, recorte, fonte, tempos, pesos, **detalhes):
    return {
        "algoritmo": algoritmo,
        "tarefa": tarefa,
        "recorte_percentual": int(recorte),
        "usuario_partida": int(fonte),
        "pesos": pesos,
        "execucoes": len(tempos),
        "tempo_medio_ms": round(mean(tempos), 4),
        "tempo_mediano_ms": round(median(tempos), 4),
        "tempos_ms": [round(t, 4) for t in tempos],
        **detalhes,
    }


SEM_PESOS = "sem pesos (cada seta é um passo)"
PESOS_DIJKSTRA = "amigo 1, inimigo 2"
SINAIS = "sinal original (amigo +1, inimigo -1)"

medicoes = []
for linha in escalabilidade_buscas.to_dict("records"):
    for algoritmo in ("BFS", "DFS"):
        medicoes.append(medicao(
            algoritmo, "escalabilidade nos recortes", linha["Recorte (%)"], linha["Usuário de partida"],
            linha[f"Tempos {algoritmo} por execução"], SEM_PESOS, alcancados=int(linha["Alcançados"]),
        ))

for linha in escalabilidade_caminhos.to_dict("records"):
    for algoritmo in ("Dijkstra", "Bellman-Ford"):
        medicoes.append(medicao(
            algoritmo, "escalabilidade nos recortes", linha["Recorte (%)"], linha["Usuário de partida"],
            linha[f"Tempos {algoritmo} por execução"], PESOS_DIJKSTRA, alcancados=int(linha["Alcançados"]),
        ))

for (origem, destino, _), linha in zip(marcacoes_piloto.itertuples(index=False, name=None), linhas_dijkstra):
    medicoes.append(medicao(
        "Dijkstra", "volta das marcações do piloto", 100, destino, linha["Tempos por execução (ms)"],
        PESOS_DIJKSTRA, destino=int(origem), custo=linha["Custo (Dijkstra)"], passos=linha["Passos (Dijkstra)"],
    ))

medicoes.append(medicao(
    "Bellman-Ford", "peso negativo sem ciclo negativo", 10, fonte_bf, tempos_bf_sem_ciclo, SINAIS,
    alcancados=len(distancias_bf), ciclo_negativo=None,
))
medicoes.append(medicao(
    "Bellman-Ford", "detecção de ciclo negativo", 100, fonte_bf, tempos_bf_com_ciclo, SINAIS,
    ciclo_negativo=[int(usuario) for usuario in ciclo_negativo], custo_ciclo=int(custo_ciclo),
))

relatorio = {
    "dataset": "soc-sign-Slashdot090221",
    "nos": grafo_completo.ordem(),
    "arcos": grafo_completo.tamanho(),
    "recortes": resumo_recortes,
    "unidade_tempo": "milissegundos",
    "medicoes": medicoes,
}
caminho_relatorio = pasta_projeto / "out" / "parte2" / "parte2_report.json"
caminho_relatorio.parent.mkdir(parents=True, exist_ok=True)
caminho_relatorio.write_text(json.dumps(relatorio, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"{len(medicoes)} medições salvas em out/parte2/parte2_report.json")
(
    pd.DataFrame(medicoes)
    .groupby(["algoritmo", "tarefa"])["tempo_medio_ms"]
    .agg(medicoes="count", tempo_medio_ms="mean")
    .round(3)
)

48 medições salvas em out/parte2/parte2_report.json


medicoes  tempo_medio_ms
algoritmo    tarefa                                                    
BFS          escalabilidade nos recortes              9          92.573
Bellman-Ford detecção de ciclo negativo               1           8.767
             escalabilidade nos recortes              9         180.164
             peso negativo sem ciclo negativo         1           0.043
DFS          escalabilidade nos recortes              9         322.278
Dijkstra     escalabilidade nos recortes              9         122.705
             volta das marcações do piloto           10         268.253